# Expectation, Variance, and Moments

| Difficulty | ███░░░░░░░ 3/10 |
| :--- | :--- |
| Prerequisites | 03.02 |
| Time | ~55 min |
| Colab GPU | Not needed |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NiravRVaghasiya/maths-for-ai/blob/main/03_Probability_and_Statistics/03_expectation_variance_moments.ipynb)

---

## 🎯 Learning Objective

Compute expectation and variance of random variables and generalize them to higher-order moments like skewness and kurtosis.

---

## 📐 Theory

A distribution (`03.02`) fully describes a random variable, but it's often too much
information. **Moments** compress a distribution into a handful of numbers that summarize its
location, spread, and shape.

### Expectation: the weighted average

The **expectation** (mean) of $X$ is the probability-weighted average of its possible values:

$$\mathbb{E}[X] = \sum_x x\,p_X(x) \quad \text{(discrete)}, \qquad \mathbb{E}[X] = \int_{-\infty}^{\infty} x\,f_X(x)\,dx \quad \text{(continuous)}.$$

$\mathbb{E}[\cdot]$ is **linear**: $\mathbb{E}[aX+bY+c] = a\mathbb{E}[X]+b\mathbb{E}[Y]+c$ for any
constants $a,b,c$ — true regardless of whether $X,Y$ are independent, and the single most-used
identity in this notebook. The **law of the unconscious statistician (LOTUS)** extends
expectation to any function $g$ of $X$ without needing $g(X)$'s own distribution:
$\mathbb{E}[g(X)] = \sum_x g(x)\,p_X(x)$.

### Variance: spread around the mean

**Variance** measures how far $X$ typically strays from its mean:

$$\text{Var}(X) = \mathbb{E}\big[(X-\mathbb{E}[X])^2\big] = \mathbb{E}[X^2] - (\mathbb{E}[X])^2,$$

the second form (expand the square and apply linearity) being the one you'll actually compute
with — it needs only $\mathbb{E}[X]$ and $\mathbb{E}[X^2]$, each a single LOTUS sum/integral.
The **standard deviation** $\sigma_X = \sqrt{\text{Var}(X)}$ returns to the original units of
$X$. Unlike expectation, variance is *not* linear: $\text{Var}(aX+b) = a^2\text{Var}(X)$ (shifts
don't affect spread; scaling does, quadratically).

### Covariance: how two variables move together

**Covariance** generalizes variance to a pair of random variables:

$$\text{Cov}(X,Y) = \mathbb{E}[(X-\mathbb{E}[X])(Y-\mathbb{E}[Y])] = \mathbb{E}[XY]-\mathbb{E}[X]\mathbb{E}[Y].$$

$\text{Cov}(X,Y)>0$ means $X,Y$ tend to be above/below their means together; $<0$ means
opposite; $\text{Cov}(X,X)=\text{Var}(X)$. Normalizing by both standard deviations gives the
scale-free **correlation** $\rho_{X,Y}=\text{Cov}(X,Y)/(\sigma_X\sigma_Y)\in[-1,1]$. Crucially,
$\text{Var}(X+Y) = \text{Var}(X)+\text{Var}(Y)+2\,\text{Cov}(X,Y)$ — variances only add when
covariance is zero, which independence guarantees (the converse is false in general: zero
covariance doesn't imply independence). This identity is why joint distributions in
[Joint, Conditional, and Marginal Distributions](04_joint_conditional_marginal.ipynb) (`03.04`)
need a full covariance *matrix*, not just a variance per variable, to describe several
variables at once.

### Moment generating functions

The **moment generating function (MGF)** packages *every* moment of $X$ into one function:

$$M_X(t) = \mathbb{E}[e^{tX}].$$

Differentiating $M_X$ and evaluating at $t=0$ recovers raw moments directly: $M_X^{(n)}(0) =
\mathbb{E}[X^n]$ (expand $e^{tX}$ as its Taylor series from `02.02`-style local approximation,
term-by-term differentiation under the expectation pulls down a factor of $X^n$ at $t=0$). MGFs
also turn a sum of independent variables into a *product* of MGFs
($M_{X+Y}(t)=M_X(t)M_Y(t)$ when $X\perp Y$), the algebraic trick behind proving the Central
Limit Theorem.

### Skewness and kurtosis: standardized higher moments

**Standardized moments** describe shape independent of location/scale. **Skewness**
$\mathbb{E}[((X-\mu)/\sigma)^3]$ measures asymmetry: zero for symmetric distributions, positive
for a distribution with a longer right tail. **Kurtosis** $\mathbb{E}[((X-\mu)/\sigma)^4]$
measures tail weight/"peakedness"; a Gaussian has kurtosis exactly 3, so **excess kurtosis**
(kurtosis $-3$) is often reported to make "heavier-tailed than Gaussian" read as a positive
number. Together, the first four standardized moments (mean, variance, skewness, kurtosis) are
a compact fingerprint of a distribution's shape.

---

In [ ]:
# Setup
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

## 👁️ Visual Intuition

Four distributions, each with mean 0 and variance 1, but visibly different shapes — skewness
and kurtosis are what distinguish them once the first two moments are pinned down identically.

In [ ]:
# Four standardized (mean=0, var=1) distributions with visibly different skewness/kurtosis
rng = np.random.default_rng(0)
n_samples = 200_000

def standardize(x):
    return (x - x.mean()) / x.std()

samples = {
    "Standard normal\n(skew=0, excess kurtosis=0)": standardize(rng.normal(size=n_samples)),
    "Right-skewed (Exponential)\n(skew>0: long right tail)": standardize(rng.exponential(size=n_samples)),
    "Left-skewed (mirrored Exponential)\n(skew<0: long left tail)": standardize(-rng.exponential(size=n_samples)),
    "Heavy-tailed (Student-t, df=3)\n(excess kurtosis>0)": standardize(rng.standard_t(df=3, size=n_samples)),
}

fig, axes = plt.subplots(1, 4, figsize=(16, 4), sharey=True)
for ax, (title, data) in zip(axes, samples.items()):
    ax.hist(data, bins=80, range=(-5, 5), density=True, color="#4C72B0", alpha=0.75)
    skew = stats.skew(data)
    kurt = stats.kurtosis(data)  # scipy reports EXCESS kurtosis (kurtosis - 3) by default
    ax.set_title(f"{title}\nskew={skew:.2f}, excess kurt={kurt:.2f}", fontsize=9)
    ax.set_xlabel("standardized value")
plt.tight_layout()
plt.show()

for title, data in samples.items():
    label = title.split("\n")[0]
    print(f"{label:38s}: mean={data.mean():+.3f}, var={data.var():.3f}, "
          f"skew={stats.skew(data):+.3f}, excess kurtosis={stats.kurtosis(data):+.3f}")

## 🐍 Implementation from Scratch

We compute $\mathbb{E}[X]$, $\text{Var}(X)$, covariance, and standardized skewness/kurtosis
directly from their defining sums/formulas (no `np.mean`/`np.var` shortcuts), verify the MGF's
derivative property numerically, and confirm everything against `scipy.stats` moment functions.

In [ ]:
# Discrete X: outcome of a loaded die, defined by an explicit PMF (not a scipy object) --
# every quantity below is computed directly from the PMF definition, LOTUS-style.
values = np.array([1, 2, 3, 4, 5, 6])
pmf = np.array([0.10, 0.10, 0.15, 0.15, 0.20, 0.30])
assert np.isclose(pmf.sum(), 1.0)

E_X = np.sum(values * pmf)                       # E[X] = sum_x x p(x)
E_X2 = np.sum(values**2 * pmf)                    # E[X^2], via LOTUS with g(x)=x^2
Var_X = E_X2 - E_X**2                             # Var(X) = E[X^2] - E[X]^2
std_X = np.sqrt(Var_X)

print(f"E[X]      = {E_X:.4f}")
print(f"E[X^2]    = {E_X2:.4f}")
print(f"Var(X)    = {Var_X:.4f}  (via E[X^2]-E[X]^2)")

# Cross-check Var(X) against the OTHER definition, E[(X-E[X])^2], to confirm they agree exactly
Var_X_direct = np.sum((values - E_X)**2 * pmf)
print(f"Var(X)    = {Var_X_direct:.4f}  (via E[(X-E[X])^2] directly -- must match)")

# Standardized skewness/kurtosis, again by LOTUS on powers of the standardized variable
skew_X = np.sum(((values - E_X) / std_X)**3 * pmf)
kurt_X = np.sum(((values - E_X) / std_X)**4 * pmf)
print(f"Skewness  = {skew_X:.4f}  (negative: this loaded die favors high faces -> left tail)")
print(f"Kurtosis  = {kurt_X:.4f}  (excess kurtosis = {kurt_X - 3:.4f})")

# Covariance/correlation between X and Y=g(X)=7-X (the "opposite face" on a standard die) --
# a deterministic decreasing function of X, so correlation should be close to -1
Y_values = 7 - values
E_Y = np.sum(Y_values * pmf)
Cov_XY = np.sum((values - E_X) * (Y_values - E_Y) * pmf)
Var_Y = np.sum((Y_values - E_Y)**2 * pmf) - 0  # already centered in the sum above
corr_XY = Cov_XY / (std_X * np.sqrt(np.sum((Y_values - E_Y)**2 * pmf)))
print(f"\nCov(X, 7-X) = {Cov_XY:.4f}, Corr(X, 7-X) = {corr_XY:.4f}  (exactly -1: Y is an exact linear function of X)")

# MGF derivative property: M_X'(0) should equal E[X], M_X''(0) should equal E[X^2] --
# verified via finite differences (central difference), since we have no symbolic M_X here
def mgf(t):
    return np.sum(np.exp(t * values) * pmf)

h = 1e-5
M_prime_0 = (mgf(h) - mgf(-h)) / (2 * h)                  # central-difference first derivative
M_double_prime_0 = (mgf(h) - 2 * mgf(0) + mgf(-h)) / h**2  # central-difference second derivative
print(f"\nM_X'(0) via finite difference  = {M_prime_0:.4f}  (should match E[X]   = {E_X:.4f})")
print(f"M_X''(0) via finite difference = {M_double_prime_0:.4f}  (should match E[X^2] = {E_X2:.4f})")

# Independent cross-check of the discrete moments against scipy.stats.rv_discrete
rv = stats.rv_discrete(name="loaded_die", values=(values, pmf))
print(f"\nscipy cross-check: mean={rv.mean():.4f}, var={rv.var():.4f}  (both match our hand computation)")

## 🤖 Why This Matters for AI

A **loss function** is an expectation: $\mathcal{L}(\theta) = \mathbb{E}_{(x,y)\sim
p_{\text{data}}}[\ell(f_\theta(x), y)]$, and since you never have access to the true data
distribution $p_{\text{data}}$, every training step estimates this expectation with a **batch
average** — a Monte Carlo estimate of $\mathbb{E}[\cdot]$, made rigorous in
[Sampling and Monte Carlo Methods](09_sampling_and_monte_carlo.ipynb) (`03.09`). **Batch
normalization** operates on the exact quantities this notebook defines: it computes each
feature's batch mean and variance and rescales activations to have mean 0, variance 1 — the
"standardize" operation used in the Visual Intuition cell above, applied inside a neural
network rather than to a plotted sample.

The specific implementation: `torch.nn.BatchNorm2d` (Ioffe & Szegedy, 2015) computes running estimates of each channel's mean and variance during training and uses them at inference time — the exact $\mu$ and $\sigma^2$ from this notebook, tracked with an exponential moving average across batches.

---

In [ ]:
# Loss as an expectation: the TRUE loss (huge validation set stand-in for "the full population")
# vs. mini-batch estimates -- averaging is a Monte Carlo estimate of E[loss], with real sampling noise
rng = np.random.default_rng(0)
n_population = 100_000
true_labels = rng.integers(0, 2, n_population)
# predicted probability for the true class -- imperfect model, some examples easy, some hard
predicted_prob_correct = np.clip(0.5 + 0.4 * (2 * true_labels - 1) * rng.uniform(0.3, 1.0, n_population)
                                  + rng.normal(scale=0.05, size=n_population), 1e-6, 1 - 1e-6)
per_example_loss = -np.log(predicted_prob_correct)  # negative log-likelihood per example (formalized in 03.07)

true_expected_loss = per_example_loss.mean()  # "population" loss -- what we actually want to minimize
print(f"True E[loss] over all {n_population:,} examples: {true_expected_loss:.4f}\n")

print("Mini-batch AVERAGES are noisy Monte Carlo estimates of that same expectation:")
for batch_size in [8, 64, 512]:
    batch_estimates = [rng.choice(per_example_loss, size=batch_size, replace=False).mean() for _ in range(5)]
    print(f"  batch_size={batch_size:4d}: 5 batch-loss estimates = {np.round(batch_estimates, 4)}, "
          f"std across batches={np.std(batch_estimates):.4f}")
print("(larger batches -> lower-variance estimates of the same true expectation -- Var(mean of n) = Var(X)/n)")

# Batch normalization: literally subtract the batch MEAN, divide by the batch STD, per feature
print("\n--- Batch normalization, computed with this notebook's own E[X]/Var(X) definitions ---")
batch = rng.normal(loc=[5.0, -2.0, 100.0], scale=[2.0, 0.5, 10.0], size=(64, 3))  # (batch, features)

batch_mean = np.array([np.sum(batch[:, j]) / batch.shape[0] for j in range(batch.shape[1])])
batch_var = np.array([np.sum((batch[:, j] - batch_mean[j])**2) / batch.shape[0] for j in range(batch.shape[1])])
eps = 1e-5
batch_normalized = (batch - batch_mean) / np.sqrt(batch_var + eps)

print(f"Per-feature batch mean (before):  {np.round(batch_mean, 3)}")
print(f"Per-feature batch var  (before):  {np.round(batch_var, 3)}")
print(f"Per-feature mean (after normalizing):  {np.round(batch_normalized.mean(axis=0), 6)}  (should be ~0)")
print(f"Per-feature var  (after normalizing):  {np.round(batch_normalized.var(axis=0), 6)}  (should be ~1)")

## 🏋️ Exercises

Six levels, in order: mechanical computation → conceptual understanding → derivation →
implementation → application → open-ended challenge. Solutions/outlines are in the collapsed
`Solution` blocks below each exercise — try before revealing.

### 🔢 COMPUTE
1. For the loaded die's PMF above (`values`/`pmf`), compute $\mathbb{E}[2X+3]$ two ways: (a)
   using linearity, $2\mathbb{E}[X]+3$, and (b) directly via LOTUS on $g(x)=2x+3$. As a second
   check, compute $\text{Var}(2X+3)$ two ways: via $a^2\text{Var}(X)$ and via
   $\mathbb{E}[(2X+3)^2]-(\mathbb{E}[2X+3])^2$ directly, confirming variance is non-negative
   both times.

<details>
<summary>💡 Solution</summary>

$\mathbb{E}[X] = \sum_x x\,p(x) = 4.15$. (a) Linearity: $2(4.15)+3 = 11.3$. (b) LOTUS directly on
$g(x)=2x+3$: $\sum_x (2x+3)p(x) = 11.3$ — identical. For variance, $\text{Var}(X)=2.8275$, so (a)
$a^2\text{Var}(X) = 2^2 \times 2.8275 = 11.31$; (b) computing
$\mathbb{E}[(2X+3)^2]-(\mathbb{E}[2X+3])^2$ directly from the PMF also gives $11.31$ — matching,
and both non-negative as required (variance is a sum of squared deviations weighted by
non-negative probabilities, so it can never be negative).

</details>

### 💭 UNDERSTAND
2. $\text{Cov}(X,Y)=0$ does **not** imply $X,Y$ are independent. Let $X\sim\text{Uniform}(-1,1)$
   and $Y=X^2$ — clearly $Y$ is a deterministic (hence maximally dependent) function of $X$.
   Without simulating anything, explain using the definition
   $\text{Cov}(X,Y)=\mathbb{E}[XY]-\mathbb{E}[X]\mathbb{E}[Y]$ why this covariance is exactly
   zero despite the strong dependence, then confirm numerically.

<details>
<summary>💡 Solution</summary>

$\mathbb{E}[X]=0$ by symmetry of the Uniform$(-1,1)$ distribution around $0$. $\mathbb{E}[XY] =
\mathbb{E}[X\cdot X^2] = \mathbb{E}[X^3]$, and $X^3$ is an *odd* function of a symmetric-around-0
random variable, so its expectation is also exactly $0$ (every positive value of $X^3$ from
$X=x>0$ is exactly canceled by the equally-likely negative value from $X=-x$). So
$\text{Cov}(X,Y) = 0 - 0\cdot\mathbb{E}[Y] = 0$. This is possible only because covariance is a
measure of *linear* association — $Y=X^2$ is a symmetric (even) function of $X$, so it has zero
*linear* trend with $X$ even though $Y$ is completely determined by $X$. Simulating
$X\sim\text{Uniform}(-1,1)$, $Y=X^2$ over a large sample gives a sample covariance very close to
$0$ (shrinking toward exactly $0$ as the sample grows), confirming the theoretical value.

</details>

### ✏️ DERIVE
3. Derive the identity $\text{Var}(X+Y) = \text{Var}(X) + \text{Var}(Y) + 2\,\text{Cov}(X,Y)$
   starting from the definition $\text{Var}(Z) = \mathbb{E}[(Z-\mathbb{E}[Z])^2]$ with
   $Z = X+Y$, using only linearity of expectation. Then specialize to independent $X,Y$ and
   state which term vanishes and why.

<details>
<summary>💡 Solution outline</summary>

By linearity, $\mathbb{E}[X+Y] = \mathbb{E}[X]+\mathbb{E}[Y]$. Let $\mu_X=\mathbb{E}[X]$,
$\mu_Y=\mathbb{E}[Y]$. Then
$$\text{Var}(X+Y) = \mathbb{E}\big[((X+Y)-(\mu_X+\mu_Y))^2\big] = \mathbb{E}\big[((X-\mu_X)+(Y-\mu_Y))^2\big].$$
Expand the square: $(X-\mu_X)^2 + (Y-\mu_Y)^2 + 2(X-\mu_X)(Y-\mu_Y)$. Taking $\mathbb{E}[\cdot]$
term-by-term (linearity again):
$$\text{Var}(X+Y) = \mathbb{E}[(X-\mu_X)^2] + \mathbb{E}[(Y-\mu_Y)^2] + 2\,\mathbb{E}[(X-\mu_X)(Y-\mu_Y)]
= \text{Var}(X) + \text{Var}(Y) + 2\,\text{Cov}(X,Y).$$
For independent $X,Y$: independence implies $\mathbb{E}[(X-\mu_X)(Y-\mu_Y)] =
\mathbb{E}[X-\mu_X]\,\mathbb{E}[Y-\mu_Y] = 0\times 0 = 0$ (expectation of a product of
independent quantities factors), so the cross term $2\,\text{Cov}(X,Y)$ vanishes and
$\text{Var}(X+Y) = \text{Var}(X)+\text{Var}(Y))$ — variances add only in this special case, which
is exactly the claim the Theory section makes.

</details>

### 🐍 IMPLEMENT
4. Generate two correlated arrays with `rng.multivariate_normal([0,0], [[1,0.7],[0.7,1]],
   size=5000)`. Compute their covariance and correlation completely from scratch (using the
   $\text{Cov}(X,Y)=\mathbb{E}[XY]-\mathbb{E}[X]\mathbb{E}[Y]$ formula, no `np.cov`), and confirm
   the correlation is close to the input value of $0.7$. As a sanity check, also verify the
   DERIVE identity numerically: confirm `Var(X+Y)` computed directly matches
   `Var(X)+Var(Y)+2*Cov(X,Y)` computed from your from-scratch pieces.

<details>
<summary>✅ How to know you're right</summary>

Your from-scratch correlation should land within about $\pm 0.03$ of $0.7$ at $n=5000$ (sampling
noise at this sample size), and should match `np.corrcoef(X, Y)[0, 1]` to within floating-point
tolerance, since both compute the identical quantity. The variance-of-sum check
(`Var(X+Y)` vs. `Var(X)+Var(Y)+2*Cov(X,Y)`) should agree to about `1e-10` or better — this is an
algebraic identity applied to the same underlying data, not an approximation, so any larger
discrepancy indicates a bug in one of the two computations rather than sampling noise.

</details>

### 🤖 APPLY
5. In the loss-as-expectation cell, increase `batch_size` through
   $\{8, 32, 128, 512, 2048\}$, drawing several batch-mean estimates at each size, and compute
   the standard deviation of the batch-loss estimate at each batch size. Plot std vs. batch size
   on a log-log plot and fit a line to `log(std)` vs. `log(batch_size)`.

<details>
<summary>✅ What you should observe</summary>

The fitted slope on the log-log plot should come out close to $-0.5$ (a representative run gives
about $-0.50$), confirming $\text{std} \propto n^{-1/2}$ — a direct consequence of
$\text{Var}(\text{mean of }n\text{ iid samples}) = \text{Var}(X)/n$, so
$\text{std}(\text{mean}) = \sigma/\sqrt{n}$. Concretely, the measured standard deviations at each
batch size should track $\sigma_{\text{pop}}/\sqrt{n}$ (using the population per-example loss's
own standard deviation) reasonably closely. Practically: to *halve* the gradient-estimate noise,
you need *four times* the batch size, not two — the $\sqrt{n}$ relationship means the return on
larger batches (in noise-reduction terms) diminishes rather than scaling linearly with compute
spent.

</details>

### 🚀 CHALLENGE
6. Skewness and kurtosis (this notebook's higher standardized moments) are themselves estimated
   from a finite sample, just like the mean in exercise 5 — but a distribution's $k$-th moment
   only exists (is finite) at all if the distribution's tails decay fast enough. For a
   Student-$t$ distribution with $\nu$ degrees of freedom, the $k$-th moment is finite if and
   only if $k < \nu$. Investigate the practical consequence: draw samples of increasing size
   ($n=50, 200, 1000, 10000, 100000$) from Student-$t$ with $\nu=10$ (skewness's 3rd moment and
   kurtosis's 4th moment both mathematically exist, since $3,4<10$) and separately from
   $\nu=3$ (the 3rd and 4th moments do **not** exist, since neither $3$ nor $4$ is $<3$). Repeat
   each sample size many times and plot the *spread* (standard deviation across repeats) of the
   sample skewness and sample excess kurtosis vs. $n$, for both $\nu$ values, alongside the
   spread of the sample mean for reference. Comment on what this implies for trusting a reported
   skewness/kurtosis number computed from a real, possibly heavy-tailed dataset.

<details>
<summary>🔍 What a strong answer covers</summary>

At $\nu=10$, all three statistics behave as expected from ordinary large-sample theory: the
spread of the sample mean, sample skewness, and sample excess kurtosis all *shrink* as $n$
grows (roughly $50\to0.17$, $200\to0.08$, ..., $100000\to0.003$ for the mean; skewness and
kurtosis shrink too, just from a larger starting spread and somewhat more slowly, since they
depend on higher, noisier powers of the data). At $\nu=3$, the sample mean still shrinks
normally (the population mean and variance are finite for any $\nu>2$) — but the sample
skewness and sample excess kurtosis do the opposite of shrinking: their spread *grows* with $n$
(in a representative run, skewness spread grows from roughly $1.5$ at $n=50$ to roughly $9$ at
$n=100{,}000$, and kurtosis spread from roughly $5$ to over $1800$), because the population's
3rd and 4th moments are actually infinite, so there is no fixed value for the sample statistic
to converge to — larger samples just have a higher chance of including an extreme draw that
swings the higher-power-weighted statistic wildly. A strong answer explicitly connects this to
the $k<\nu$ existence condition rather than treating it as numerical noise, and draws the
practical conclusion: a headline "skewness" or "kurtosis" number computed from real, plausibly
heavy-tailed data is only trustworthy if you have reason to believe the underlying distribution
actually has a finite moment of that order — otherwise more data doesn't make the estimate more
reliable, it can make it *more* erratic, which is the opposite of what "more data is better"
usually means.

</details>

---

## 📚 Further Reading
- Wasserman, *All of Statistics*, Ch. 3–4 (Expectation, Variance, and Moments)
- Ioffe & Szegedy, ["Batch Normalization: Accelerating Deep Network Training by Reducing Internal Covariate Shift"](https://arxiv.org/abs/1502.03167)
- Casella & Berger, *Statistical Inference*, Ch. 2 (Transformations and Expectations)

---

*Next notebook: [Joint, Conditional, and Marginal Distributions](04_joint_conditional_marginal.ipynb)*